# 001 Weather Assistant Agent

这是智能体实战目录的第一份 Notebook。

学习目标：

1. 把已有的 `weather-query-assistant` Skill 接到一个业务智能体里
2. 理解 Skill 和 Tool 在真实流程里的分工
3. 做一个最小可运行的天气助手 Agent
4. 学会用“意图识别 -> 工具调用 -> 结果整理”的方式拆解智能体

这份 Notebook 会复用仓库里的真实 Skill，而不是重新写一套孤立示例。

## 先理解这份实战的边界

第一版天气助手只做四件事：

1. 判断用户是不是在问天气
2. 提取地点
3. 调用天气查询工具
4. 把工具结果整理成简洁中文回答

暂时不做这些内容：

- 多城市复杂行程规划
- 长期气候分析
- 用户画像记忆
- 多 Agent 协作

第一版先把业务闭环跑通。

## Skill 和 Tool 在这里怎么分工

这份实战里有两个层次：

- `weather-query-assistant` Skill：告诉智能体查天气应该怎么做，比如先规范化地点、优先用 `wttr.in`、必要时回退到 Open-Meteo。
- `fetch_weather.py` Tool：真正执行天气查询，并返回结构化结果。

你可以把它类比成 Java 项目：

- Skill 像团队 SOP 和领域规范
- Tool 像 `WeatherClient#getCurrentWeather(location)` 这种可调用函数

## 整体流程

这份 Notebook 的流程如下：

```text
用户问题
  -> 模型识别天气意图
  -> 应用代码选择工具
  -> fetch_weather.py 查询天气
  -> 模型整理最终回答
  -> 返回完整调试信息
```

## 加载环境变量

这里沿用前面 OpenAI Notebook 的方式：自动向上查找项目根目录 `.env`。

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "app").exists():
            return path
    raise RuntimeError("没有找到项目根目录，请确认当前 Notebook 在 fastapi-study 仓库内运行")


def load_project_env() -> Path | None:
    root = find_project_root()
    env_path = root / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        return env_path
    return None


PROJECT_ROOT = find_project_root()
env_path = load_project_env()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print("PROJECT_ROOT =", PROJECT_ROOT)
print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)

## 创建 OpenAI 客户端

这份实战继续使用 `Chat Completions API` 和 `beta.chat.completions.parse`，保持和前面几份 Notebook 的学习方式一致。

In [ ]:
from openai import OpenAI


if not OPENAI_API_KEY:
    raise ValueError("请先在项目根目录 .env 中配置 OPENAI_API_KEY")


client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)

## 接入已有天气 Tool

这里不是重新实现天气查询，而是把仓库独立 Tool 目录加到 Python import 路径。

这一点很关键：Skill 负责说明流程，Tool 负责执行能力，两者不要混在同一个目录。

In [ ]:
import sys


WEATHER_SKILL_DIR = PROJECT_ROOT / ".agents" / "skills" / "weather-query-assistant"
WEATHER_TOOL_DIR = PROJECT_ROOT / ".agents" / "tools" / "weather"

if str(WEATHER_TOOL_DIR) not in sys.path:
    sys.path.insert(0, str(WEATHER_TOOL_DIR))

from fetch_weather import fetch_weather
from normalize_location import normalize_location


print("Weather skill dir =", WEATHER_SKILL_DIR)
print("Weather tool dir =", WEATHER_TOOL_DIR)
print("Weather tool loaded =", fetch_weather.__name__)

## 先单独测试天气工具

在接模型之前，先确认工具函数本身能工作。

这一步很像后端开发里先测 `WeatherClient`，再把它接进 service。

In [ ]:
def fake_weather_http_get(url: str, timeout: float) -> tuple[int, str]:
    if "wttr.in" in url:
        return 200, "Beijing: Sunny +25C 40% 9km/h"
    return 200, '{"current_weather":{"temperature":21.5,"windspeed":8.0,"weathercode":2}}'


tool_smoke_result = fetch_weather("beijing", http_get_func=fake_weather_http_get)
tool_smoke_result

## 第一步：定义天气意图对象

这里先让模型把用户自然语言转成稳定字段。

这一步对应 Java 里的 DTO：先把不稳定的文本请求变成程序可判断的数据结构。

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class WeatherIntent(BaseModel):
    intent: Literal["current_weather", "forecast", "general_chat", "unknown"] = Field(
        description="用户意图类型"
    )
    location: str | None = Field(default=None, description="天气查询地点，例如 Beijing、Shanghai、New York")
    needs_tool: bool = Field(description="是否需要调用天气工具")
    clarification_question: str | None = Field(
        default=None,
        description="当缺少地点或问题不清楚时，给用户的澄清问题"
    )

## 第二步：定义意图识别提示词

天气助手第一版只需要识别：

- 当前天气
- 简单预报
- 普通聊天
- 无法判断

如果用户问天气但没有给地点，就不要猜，应该要求澄清。

In [ ]:
INTENT_SYSTEM_PROMPT = """
你是一个天气助手的意图识别器。

请根据用户输入识别以下字段：
1. intent:
   - current_weather: 查询当前天气、温度、风、湿度等
   - forecast: 查询今天/明天/未来几天的简单天气趋势
   - general_chat: 普通聊天或不需要调用天气工具的问题
   - unknown: 无法判断
2. location: 如果用户给了地点，就提取地点；如果没有地点，返回 null
3. needs_tool: 只要需要实时天气数据，就返回 true
4. clarification_question: 如果用户需要天气但没给地点，就生成一句简短澄清问题

要求：
- 不要编造地点
- 不要编造天气数据
- 回答时严格遵守 schema
""".strip()

## 第三步：封装意图识别函数

这一步使用结构化输出，让后面的工具路由更稳定。

In [ ]:
def parse_weather_intent(message: str) -> WeatherIntent:
    completion = client.beta.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
        response_format=WeatherIntent,
    )

    choice = completion.choices[0]
    refusal = getattr(choice.message, "refusal", None)
    if refusal:
        raise ValueError(f"模型拒绝回答: {refusal}")

    parsed = choice.message.parsed
    if parsed is None:
        raise ValueError("模型没有返回可解析的天气意图结果")

    return parsed

## 先试一下意图识别

这一步只看模型是否能把问题拆成稳定字段，还不查真实天气。

In [ ]:
intent_examples = [
    "北京现在天气怎么样？",
    "明天上海会下雨吗？",
    "今天适合穿什么？",
    "你好，介绍一下你自己",
]

for message in intent_examples:
    parsed = parse_weather_intent(message)
    print("user:", message)
    print("intent:", parsed.model_dump())
    print("-" * 60)

## 第四步：把意图映射到天气工具

第一版不用让模型自己决定调用哪个函数。

我们先采用更容易理解和调试的方式：模型只负责识别意图，应用代码负责路由到工具。

In [ ]:
def get_current_weather(location: str) -> dict:
    return fetch_weather(location, source="auto", units="metric")


TOOL_BY_INTENT = {
    "current_weather": get_current_weather,
    "forecast": get_current_weather,
}

## 第五步：执行工具并拿到结构化结果

这里要处理三个常见分支：

1. 不需要工具：直接返回 `None`
2. 需要工具但没有地点：返回澄清问题
3. 有地点：调用天气工具

In [ ]:
def execute_weather_tool(parsed_intent: WeatherIntent) -> dict | None:
    if not parsed_intent.needs_tool:
        return None

    if not parsed_intent.location:
        return {
            "ok": False,
            "error": "missing_location",
            "clarification_question": parsed_intent.clarification_question or "你想查询哪个城市的天气？",
        }

    tool_func = TOOL_BY_INTENT.get(parsed_intent.intent)
    if tool_func is None:
        return {"ok": False, "error": f"当前意图 {parsed_intent.intent} 没有对应工具"}

    return tool_func(parsed_intent.location)

## 先测试“意图 -> 工具结果”

这一步会调用真实天气工具。`wttr.in` 不需要 API Key；如果主路径失败，工具会尝试 Open-Meteo fallback。

In [ ]:
message = "北京现在天气怎么样？"
parsed_intent = parse_weather_intent(message)
tool_result = execute_weather_tool(parsed_intent)

print("parsed_intent =", parsed_intent.model_dump())
print("tool_result =", tool_result)

## 第六步：定义最终回答提示词

工具已经负责拿数据了。

最后这一步，模型只负责把结构化结果讲清楚。

In [ ]:
ANSWER_SYSTEM_PROMPT = """
你是一个天气助手。
请根据用户问题、识别出的意图和天气工具结果，用简洁中文生成最终回答。

要求：
1. 不要编造工具结果中不存在的天气数据
2. 如果工具结果 ok=false，要说明原因或提出澄清问题
3. 如果用户问穿衣建议，只能基于工具返回的天气信息给保守建议
4. 如果没有工具结果且属于普通聊天，可以直接回答
5. 回答尽量短，优先给地点、天气、温度、风或湿度等关键信息
""".strip()

## 第七步：封装最终回答函数

这里把用户问题、意图和工具结果一起交给模型，让模型负责表达。

In [ ]:
import json


def render_weather_answer(user_message: str, parsed_intent: WeatherIntent, tool_result: dict | None) -> str:
    prompt = {
        "user_message": user_message,
        "parsed_intent": parsed_intent.model_dump(),
        "tool_result": tool_result,
    }

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)},
        ],
    )
    return response.choices[0].message.content or ""

## 第八步：封装最小天气助手 Agent

现在把整个流程串起来。

返回值里保留 `parsed_intent` 和 `tool_result`，是为了方便你调试智能体每一步到底发生了什么。

In [ ]:
def weather_agent_chat(user_message: str) -> dict:
    parsed_intent = parse_weather_intent(user_message)
    tool_result = execute_weather_tool(parsed_intent)
    final_answer = render_weather_answer(user_message, parsed_intent, tool_result)

    return {
        "user_message": user_message,
        "parsed_intent": parsed_intent.model_dump(),
        "tool_result": tool_result,
        "final_answer": final_answer,
    }

## 试几个完整问题

这一步是本 Notebook 的验收点。

你应该能看到：

- 普通天气问题会触发工具
- 没有地点的问题会触发澄清
- 普通聊天不会查天气

In [ ]:
questions = [
    "北京现在天气怎么样？",
    "上海今天适合穿什么？",
    "今天会下雨吗？",
    "你好，你能做什么？",
]

for question in questions:
    result = weather_agent_chat(question)
    print("user:", result["user_message"])
    print("intent:", result["parsed_intent"])
    print("tool_result:", result["tool_result"])
    print("final_answer:", result["final_answer"])
    print("=" * 80)

## 这个版本和真实业务的距离

现在你已经有了一个最小天气助手：

```text
结构化意图 -> 工具调用 -> 结果整理
```

下一步如果要接进 FastAPI，通常会这样拆：

- `schema`：定义请求和响应 DTO
- `service`：放 `weather_agent_chat` 这类业务编排
- `router`：提供 `/api/v1/agents/weather/chat` 接口
- `tests`：覆盖缺少地点、工具失败、正常回答三个场景

先不要急着上复杂 Agent 框架。这个版本已经足够说明业务 Agent 的核心链路。

## 本阶段小结

这一份 Notebook 完成了三件事：

1. 复用了已有 `weather-query-assistant` Skill
2. 使用 `.agents/tools/weather/fetch_weather.py` 作为可调用 Tool
3. 做出一个最小可运行的天气助手 Agent

你现在可以把智能体理解成一条非常普通的业务链路：

```text
用户输入 -> 参数识别 -> 调业务工具 -> 整理响应
```

模型不是替代后端代码，而是负责处理自然语言和最终表达；稳定、可测试的能力仍然应该沉到工具和 service 层。